# When Crime Calls, Does Help Follow?
### Exploring how violent crime and EMS response times overlap in San Francisco 

**Group 47 — 02806 Social Data Analysis and Visualization**

**Explainer notebook for Project Assignment B**

**Authors:**  

Ignacio Ripoll González - s242875

Raquel Ruiz Pascual - s254628

---

This notebook explains the work behind our web data story. It includes the motivation for the project, the datasets we used, the main cleaning and preprocessing steps, the exploratory analysis, the comparison between crime levels and EMS response times, and a reflection on the visualizations we chose.


## 1. Motivation

### 1.1 What are the datasets?

We work with two large open datasets from the City and County of San Francisco, published through the DataSF Open Data Portal:

| Dataset | Source | Rows | Time range |
|---|---|---|---|
| **SF Crime** | DataSF: Police Department Incident Reports | ~2.8 M | 2003-2025 |
| **SF Fire / EMS Calls** | DataSF: Fire Department Calls for Service | ~7.3 M | 2000-2026 |

The **crime dataset** was created during Week 2 of the course by joining two San Francisco crime datasets: one historical dataset from 2003-2018, and one newer dataset from 2018 onwards. Since the city changed the way it records crime data in 2018, the two files did not have the same columns or category names. In Week 2, these differences were cleaned so that both periods could be used together. The preprocessing done in Week 2 to merge these two datasets is not included here, as we only use the resulting merged dataset.

For this project, we only keep **Assault** and **Robbery**, because they are the violent crime types most relevant to our question. The resulting crime dataset contains information such as incident type, police district, date, time, and location.

The **Fire/EMS dataset** comes from San Francisco’s Computer-Aided Dispatch system and contains dispatched unit responses to 911 calls, including Medical Incidents that require EMS staff. According to the DataSF documentation, the dataset is response-based. This means that one emergency call can generate multiple records if several units respond. Therefore, when we count rows in the Fire/EMS dataset, we are counting unit responses rather than unique emergency calls.

This dataset also includes timestamps for different stages of the response process, such as when a unit was dispatched and when it arrived on scene.

The dataset references are included at the end of the notebook.

### 1.2 Why these datasets?

San Francisco is a useful case study because it is a city with strong contrasts between neighborhoods. Some areas, such as the Tenderloin, Mission, and Bayview, experience higher levels of violent crime than others. These same areas may also depend heavily on fast emergency responses.

Our central research question is:

> **Do neighborhoods with higher violent crime levels experience different EMS response times?**

We also examine how EMS response times vary across the city, and whether some neighborhoods appear to experience slower responses than others.

This question matters because police incidents and medical emergencies are usually studied separately. By comparing them across neighborhoods and time, we can explore whether emergency response is evenly distributed, or whether some areas face longer delays on top of higher crime levels.

### 1.3 Goal for the end user's experience

Our target audience is a non-technical reader, for example someone who lives in San Francisco or is interested in urban safety. We want the reader to understand:

1. **Where** violent crime is more concentrated in the city and **when** it peaks
2. **How** EMS response times vary between neighborhoods
3. **Whether** there is a relationship between crime pressure and emergency response performance
4. **Why** this matters for fairness across neighborhoods and public safety

In [ ]:
# ============================================================
# Imports and setup
# This first cell loads the libraries and defines values that will be reused later
# ============================================================

# Libraries for data handling, interactive plots, statistics and regression, text parsing, and warning control
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import folium
from folium.plugins import HeatMap

from scipy import stats
import statsmodels.api as sm

import re
import warnings

# Some setups
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

# Colors used in the plots, we keep them consistent so the story is easier to follow
COLOR_CRIME    = '#e94560'   # red = violent crime
COLOR_EMS      = '#1a6b9a'   # blue = Fire/EMS medical calls
COLOR_RESPONSE = '#f5a623'   # amber = response time
TEMPLATE       = 'plotly_white'

# Main time window used in the final analysis
# The full datasets cover more years, but the filtered project subsets use 2016-2025
# The end date is 2026-01-01 because we use it as an exclusive limit, so it includes all of 2025
ANALYSIS_START = '2016-01-01'
ANALYSIS_END   = '2026-01-01'

# Grid size used later when we need a common spatial unit for crime and Fire/EMS data
# A value of 0.01 degrees is approximately 1 km in San Francisco
GRID_SIZE = 0.01

# Fixed order for weekday plots, instead of alphabetical order
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Helper function used later to place each record into a spatial grid cell
def assign_grid(df, lat_col='Latitude', lon_col='Longitude', gs=GRID_SIZE):
    out = df.copy()
    out['lat_bin'] = (out[lat_col] // gs) * gs
    out['lon_bin'] = (out[lon_col] // gs) * gs
    out['grid_id'] = (out['lat_bin'].round(4).astype(str)
                      + '_' + out['lon_bin'].round(4).astype(str))
    return out

# Helper function used to convert Fire/EMS location strings into latitude and longitude
# The Fire/EMS dataset stores locations as WKT POINT strings, for example: POINT (-122.41 37.78)
def extract_point(s):
    if pd.isna(s):
        return pd.Series([np.nan, np.nan])
    m = re.search(r'POINT \(([-\d.]+) ([-\d.]+)\)', str(s))
    if m:
        return pd.Series([float(m.group(2)), float(m.group(1))])  # latitude, longitude
    return pd.Series([np.nan, np.nan])


print('Environment ready.')


Environment ready.


---
## 2. Basic Statistics and Preprocessing

### 2.1 Dataset overview

Before applying any filters, we first looked at both datasets as a whole. This helped us understand their **size, time range, main variables**, and possible data quality issues such as **missing values**.

This first overview also helped us decide which variables were useful for the project, especially the crime category, call type, timestamps, response time fields, and geographic location.

The basic distributions of crime categories and Fire/EMS call types are shown below. This gives a first idea of the size of our final subsets compared with the full datasets.

In [ ]:
# ============================================================
# Load the two datasets
# The Fire/EMS file is large, so this cell can take a few minutes to run
# ============================================================

df_crime = pd.read_csv('data/SF_Crime_Merged_2003_2025.csv')
df_ems   = pd.read_csv('data/SF_EMSCalls_2000_2026.csv', low_memory=False)

# Quick overview of the full datasets before filtering
print('=== Dataset overview ===')
overview = pd.DataFrame({
    'Dataset'           : ['SF Crime', 'SF Fire/EMS'],
    'Rows'              : [f'{len(df_crime):,}', f'{len(df_ems):,}'],
    'Columns'           : [df_crime.shape[1], df_ems.shape[1]],
    'File size (approx)': ['277 MB', '3.3 GB'],
    'Years covered'     : ['2003-2025', '2000-2026'],
})
display(overview)

# Check which crime columns have the most missing values
print('\nCrime — top missing columns:')
display(
    df_crime.isna().mean()
    .sort_values(ascending=False)
    .head(5)
    .rename('missing_ratio')
    .to_frame()
)

# Check which Fire/EMS columns have the most missing values
print('\nEMS — top missing columns:')
display(
    df_ems.isna().mean()
    .sort_values(ascending=False)
    .head(10)
    .rename('missing_ratio')
    .to_frame()
)


In [ ]:
# ============================================================
# First look at the main categories in both datasets
# This is done before filtering, to understand what the full datasets contain
# ============================================================

# Count how many crime incidents belong to each broad crime category
top_cat = (
    df_crime['Super Category']
    .value_counts()
    .reset_index()
    .rename(columns={'Super Category': 'category', 'count': 'n'})
)

# Plot the 10 most common broad crime categories
fig_cat = px.bar(
    top_cat.head(10),
    x='n', y='category',
    orientation='h',
    title='Crime Incidents by Super Category — Full Dataset (2003-2025)',
    labels={'n': 'Incidents', 'category': ''},
    color_discrete_sequence=[COLOR_CRIME],
    template=TEMPLATE,
)
fig_cat.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_cat.show()


# Count how many Fire/EMS records belong to each call type
ems_types = (
    df_ems['Call Type']
    .value_counts()
    .reset_index()
    .rename(columns={'Call Type': 'call_type', 'count': 'n'})
)

# Plot the 10 most common Fire/EMS call types
fig_ems_t = px.bar(
    ems_types.head(10),
    x='n', y='call_type',
    orientation='h',
    title='Top 10 EMS Call Types — Full Dataset (2000-2026)',
    labels={'n': 'Calls', 'call_type': ''},
    color_discrete_sequence=[COLOR_EMS], 
    template=TEMPLATE,
)
fig_ems_t.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_ems_t.show()

# Check how large our main categories are compared with the full datasets
violent_share = (
    df_crime['Super Category']
    .value_counts(normalize=True)
    .get('Violent Crime', 0) * 100
)
medical_share = (
    df_ems['Call Type']
    .value_counts(normalize=True)
    .get('Medical Incident', 0) * 100
)

print(f'Violent crimes represent {violent_share:.1f}% of all SF crime incidents.')
print(f'Medical Incidents represent {medical_share:.1f}% of all EMS calls.')
print('These two subsets form the basis of the analysis.')


Violent crimes represent 10.6% of all SF crime incidents.
Medical Incidents represent 66.1% of all EMS calls.
These two subsets form the basis of the analysis.


### 2.2 Data cleaning and preprocessing

Before starting the analysis, we **filtered** both datasets to keep only records that were relevant for our research question. 

- **Crime dataset:**

We focus on **Assault** and **Robbery**. These are violent crime categories that are more likely to be connected to situations where urgent medical help may be needed, mainly because of possible acute injuries. Other categories in the dataset, such as property, financial, or drug-related crimes, are less directly related to urgent EMS responses.

The full datasets cover a longer period, but for the final analysis we use **2016-2025**. This gave us a cleaner shared window for comparing the crime data with the Fire/EMS response data. The year **2026** is excluded because it is only a partial year in the Fire/EMS dataset and is not covered in the same way by the crime dataset.

- **EMS dataset:**

For the Fire/EMS data, we only keep records where the call type is **Medical Incident**. This removes other calls such as structure fires, alarms, traffic collisions, or other non-medical calls from the analysis. By doing this, we focus on emergency responses that are more relevant to injuries or medical emergencies.

Our main response time metric is **`dispatch_to_on_scene_min`**. It measures the number of minutes between when a unit was dispatched and when it arrived on scene. We use this instead of the full time from call received to arrival (also called *received-to-on-scene*) because it focuses more on the response and travel part of the process. This is the part of the delay that is more directly controlled by the Fire Department, while call-center processing is excluded.

Response values outside the range **0-60 minutes** are treated as extreme values and removed. Since EMS responses are normally expected to arrive much faster than one hour, these values are likely to be outliers or non-representative cases.

- **Geographic coordinates:**

The Fire/EMS dataset stores location as a WKT POINT string, for example `POINT (-122.41 37.78)`. We parse this into **separate Latitude and Longitude columns** so that the records can be mapped spatially.

The crime dataset already includes separate Latitude and Longitude columns, so we could use them directly after filtering.


In [ ]:
# ============================================================
# Crime data preprocessing
# We create the crime subset used in the project
# ============================================================

# Convert the incident date and time column into a datetime format
df_crime['Incident Datetime'] = pd.to_datetime(df_crime['Incident Datetime'], errors='coerce')

# Keep only the final analysis period, 2016-2025
df_crime_project = df_crime[
    (df_crime['Incident Datetime'] >= ANALYSIS_START) &
    (df_crime['Incident Datetime'] <  ANALYSIS_END)
].copy()

# Keep only the violent crime types used in our research question
VIOLENT_FOCUS = ['Assault', 'Robbery']
df_crime_project = df_crime_project[
    df_crime_project['Focus Crime'].isin(VIOLENT_FOCUS)
].copy()

# Create time variables that will be useful for the temporal analysis
df_crime_project['year']        = df_crime_project['Incident Datetime'].dt.year
df_crime_project['month']       = df_crime_project['Incident Datetime'].dt.to_period('M').astype(str)
df_crime_project['hour']        = df_crime_project['Incident Datetime'].dt.hour
df_crime_project['day_of_week'] = df_crime_project['Incident Datetime'].dt.day_name()

# Check the final crime subset
print('Crime project subset:', df_crime_project.shape)
print('\nFocus crime breakdown:')
print(df_crime_project['Focus Crime'].value_counts())
print('\nDate range:',
      df_crime_project['Incident Datetime'].min(),
      '→',
      df_crime_project['Incident Datetime'].max())
display(df_crime_project.head(3))


Crime project subset: (110062, 12)

Focus crime breakdown:
Focus Crime
Assault    82056
Robbery    28006
Name: count, dtype: int64

Date range: 2016-01-01 00:01:00 → 2025-12-31 23:57:00


,Incident Datetime,Incident Category,Police District,Latitude,Longitude,Super Category,Year,Focus Crime,year,month,hour,day_of_week
1621114,2016-01-01 00:01:00,ASSAULT,CENTRAL,37.788,-122.408,Violent Crime,2016,Assault,2016,2016-01,0,Friday
1621122,2016-01-01 00:01:00,ASSAULT,NORTHERN,37.786,-122.420,Violent Crime,2016,Assault,2016,2016-01,0,Friday
1621137,2016-01-01 00:09:00,ASSAULT,MISSION,37.766,-122.422,Violent Crime,2016,Assault,2016,2016-01,0,Friday


In [ ]:
# ============================================================
# Fire/EMS data preprocessing
# We create the Medical Incident response subset used in the project
# This cell can take a few minutes because the dataset is very large and parsing 8 datetime columns on 7.3 M rows takes time
# ============================================================

# Convert all relevant Fire/EMS timestamp columns into datetime format
EMS_TIME_COLS = [
    'Received DtTm', 'Entry DtTm', 'Dispatch DtTm',
    'Response DtTm', 'On Scene DtTm',
    'Transport DtTm', 'Hospital DtTm', 'Available DtTm'
]
for col in EMS_TIME_COLS:
    df_ems[col] = pd.to_datetime(df_ems[col], errors='coerce')

# Extract latitude and longitude from the location string
df_ems[['Latitude', 'Longitude']] = df_ems['case_location'].apply(extract_point)

# Keep only the final analysis period, 2016-2025
df_ems_project = df_ems[
    (df_ems['Received DtTm'] >= ANALYSIS_START) &
    (df_ems['Received DtTm'] <  ANALYSIS_END)
].copy()

# Keep only Medical Incident unit responses
df_ems_project = df_ems_project[
    df_ems_project['Call Type'] == 'Medical Incident'
].copy()

# Calculate response time metrics in minutes
# The main metric is dispatch_to_on_scene_min, from dispatch to arrival on scene
df_ems_project['dispatch_to_on_scene_min'] = (
    (df_ems_project['On Scene DtTm'] - df_ems_project['Dispatch DtTm'])
    .dt.total_seconds() / 60
)
df_ems_project['received_to_on_scene_min'] = (
    (df_ems_project['On Scene DtTm'] - df_ems_project['Received DtTm'])
    .dt.total_seconds() / 60
)

# Remove response times that are negative or longer than one hour, since they are likely outliers
df_ems_project = df_ems_project[
    df_ems_project['dispatch_to_on_scene_min'].between(0, 60)
].copy()

# Create time variables that will be useful for the temporal analysis
df_ems_project['year']        = df_ems_project['Received DtTm'].dt.year
df_ems_project['month']       = df_ems_project['Received DtTm'].dt.to_period('M').astype(str)
df_ems_project['hour']        = df_ems_project['Received DtTm'].dt.hour
df_ems_project['day_of_week'] = df_ems_project['Received DtTm'].dt.day_name()

# Check the final Medical Incident response subset
print('EMS project subset:', df_ems_project.shape)
print('\nResponse time summary (dispatch_to_on_scene_min):')
display(df_ems_project['dispatch_to_on_scene_min'].describe())
print('\nTop 5 neighborhoods by EMS call volume:')
print(df_ems_project['Neighborhooods - Analysis Boundaries'].value_counts().head(5))
display(df_ems_project.head(3))


EMS project subset: (1833801, 44)

Response time summary (dispatch_to_on_scene_min):


count   1833801.000
mean          7.891
std           6.164
min           0.000
25%           3.933
50%           5.850
75%          10.000
max          60.000
Name: dispatch_to_on_scene_min, dtype: float64


Top 5 neighborhoods by EMS call volume:
Neighborhooods - Analysis Boundaries
Tenderloin                        298598
South of Market                   203696
Mission                           177909
Financial District/South Beach    116324
Bayview Hunters Point              96434
Name: count, dtype: int64


,Call Number,Unit ID,Incident Number,Call Type,Call Date,Watch Date,Received DtTm,Entry DtTm,Dispatch DtTm,Response DtTm,On Scene DtTm,Transport DtTm,Hospital DtTm,Call Final Disposition,Available DtTm,Address,City,Zipcode of Incident,Battalion,Station Area,Box,Original Priority,Priority,Final Priority,ALS Unit,Call Type Group,Number of Alarms,Unit Type,Unit sequence in call dispatch,Fire Prevention District,Supervisor District,Neighborhooods - Analysis Boundaries,RowID,case_location,data_as_of,data_loaded_at,Latitude,Longitude,dispatch_to_on_scene_min,received_to_on_scene_min,year,month,hour,day_of_week
0,160943727,53,16037460,Medical Incident,04/03/2016,04/03/2016,2016-04-03 23:15:12,2016-04-03 23:18:05,2016-04-03 23:18:33,2016-04-03 23:18:45,2016-04-03 23:35:10,2016-04-03 23:46:08,2016-04-04 00:11:46,Code 2 Transport,2016-04-04 00:47:29,POLK ST/CEDAR ST,San Francisco,94109.000,B04,03,3121,2,2,2,True,Non Life-threatening,1,MEDIC,2.000,4.000,6.000,Tenderloin,160943727-53,POINT (-122.41983 37.786358),2024 Feb 05 03:27:52 AM,2024 Feb 05 10:56:25 AM,37.786,-122.420,16.617,19.967,2016,2016-04,23,Sunday
1,161021964,AM14,16040565,Medical Incident,04/11/2016,04/11/2016,2016-04-11 13:14:47,2016-04-11 13:19:53,2016-04-11 13:20:42,2016-04-11 13:21:06,2016-04-11 13:27:48,2016-04-11 14:14:47,2016-04-11 14:33:36,Code 2 Transport,2016-04-11 15:06:20,VALENCIA ST/15TH ST,San Francisco,94103.000,B02,06,5226,2,2,2,False,Potentially Life-Threatening,1,PRIVATE,1.000,2.000,9.000,Mission,161021964-AM14,POINT (-122.42204 37.76654),2024 Feb 05 03:27:52 AM,2024 Feb 05 10:56:25 AM,37.767,-122.422,7.100,13.017,2016,2016-04,13,Monday
3,160931745,E03,16036856,Medical Incident,04/02/2016,04/02/2016,2016-04-02 13:08:02,2016-04-02 13:09:44,2016-04-02 13:10:16,2016-04-02 13:13:09,2016-04-02 13:15:07,NaT,NaT,Against Medical Advice,2016-04-02 13:16:51,WILLOW ST/LARKIN ST,San Francisco,94109.000,B02,03,1643,3,3,3,True,Potentially Life-Threatening,1,ENGINE,2.000,2.000,6.000,Tenderloin,160931745-E03,POINT (-122.41762 37.78378),2024 Feb 05 03:27:52 AM,2024 Feb 05 10:56:25 AM,37.784,-122.418,4.850,7.083,2016,2016-04,13,Saturday


In [ ]:
# ============================================================
# Track how much data remains after filtering
# This shows how the full datasets become the smaller project subsets
# ============================================================

tracking = pd.DataFrame({
    'Dataset'              : ['Crime', 'EMS'],
    'Rows (original)'      : [len(df_crime), len(df_ems)],
    'Rows (project subset)': [len(df_crime_project), len(df_ems_project)],
})

# Count how many rows were removed by the filters
tracking['Rows removed'] = (
    tracking['Rows (original)'] - tracking['Rows (project subset)']
)

# Calculate the percentage of rows kept in each dataset
tracking['% kept'] = (
    100 * tracking['Rows (project subset)'] / tracking['Rows (original)']
).round(2)

# Display the final before and after comparison
print('Dataset size tracking:')
display(tracking)
print(
    '\nWe retain ~4% of crime rows (Assault + Robbery, 2016-2025) '
    'and ~25% of EMS rows (Medical Incidents, 2016-2025 with valid response times).'
)


Dataset size tracking:


,Dataset,Rows (original),Rows (project subset),Rows removed,% kept
0,Crime,2811881,110062,2701819,3.910
1,EMS,7289666,1833801,5455865,25.160



We retain ~4% of crime rows (Assault + Robbery, 2016–2025) and ~25% of EMS rows (Medical Incidents, 2016–2025 with valid response times).


### 2.3 Final project datasets for analysis

After filtering, the final project datasets used in the analysis contain:

- **110,062 Assault and Robbery incidents** from **2016-2025** -> around **3.9%** of the original crime dataset
- **1,833,801 Medical Incident unit responses** from **2016-2025** -> around **25.2%** of the original Fire/EMS dataset

This reduction is expected, because we are not analysing all public safety activity in the city. We only keep the crime types and Fire/EMS responses that are most relevant to our research question.

It is also important to remember that the Fire/EMS number refers to **unit responses**, not unique emergency calls. One emergency call can involve several units, so the same call number can appear more than once in the dataset.

The full crime dataset contains around **285,000 Assault and Robbery incidents** across the longer **2003-2025** period. However, the final analysis in this notebook focuses on **2016-2025**, which gives us a cleaner shared period for comparing crime and Fire/EMS response data.


### 2.4 Key points from the exploratory analysis

The initial dataset overview is summarized below:

| Dataset | Rows | Columns | File size | Years covered |
|---|---:|---:|---:|---|
| SF Crime | 2,811,881 | 8 | 277 MB | 2003-2025 |
| SF Fire/EMS | 7,289,666 | 36 | 3.3 GB | 2000-2026 |

Some key points from the first exploration are:

- Both datasets are large, especially the Fire/EMS dataset. It has more than **7.3 million rows** and its file size is around **3.3 GB**, so some parts of the notebook take a long time to run

- The missing-value checks show that the main variables we use for the analysis are usable after filtering. In the crime dataset, the relevant variables such as date, category, police district, and location are mostly complete. In the Fire/EMS dataset, missing values appear mostly in later stages such as **Transport DtTm** or **Hospital DtTm**, which makes sense because not every response involves hospital transport

- In the crime dataset, **Property Crime** is the largest group. **Violent Crime** represents **10.6%** of all crime incidents, so our crime subset is much smaller than the full dataset

- In the Fire/EMS dataset, **Medical Incident** is the largest call type, with **66.1%** of all Fire/EMS records. This supports our choice of using Medical Incidents for the analysis

- After preprocessing, the crime subset contains **82,056 Assaults** and **28,006 Robberies**. Assault is therefore the main part of our violent crime subset

- The Fire/EMS subset contains **1,833,801 Medical Incident unit responses**. The mean response time is around **7.9 minutes**, while the median is around **5.9 minutes**. This means that more than half of the responses are faster than the average, but some long response times increase the mean

- The highest Medical Incident volumes appear in **Tenderloin**, **South of Market**, **Mission**, **Financial District/South Beach**, and **Bayview Hunters Point**. These neighborhoods are therefore important for the spatial part of the analysis

---

## 3. Data Analysis

### 3.1 How we compare the datasets

Before comparing crime and EMS response times, it is important to clarify that the two datasets are **not directly linked by a common event ID**. This means that we do not match individual crime incidents to individual EMS responses.

Instead, we compare both datasets after grouping them by time and by common spatial units, such as neighborhoods or grid cells. This approach allows us to study whether areas and periods with higher violent crime levels also tend to show different EMS response times.

We first explore each dataset separately:

- For **violent crime**, we look at how incidents change over time, when they are most common during the week, and where they are concentrated in the city

- For **Fire/EMS data**, we focus on Medical Incident unit responses and analyse how dispatch-to-on-scene response time varies by year, hour, day of week, and neighborhood

The analysis follows four main steps:

1. **Temporal patterns:** how violent crime and EMS response time change by year, hour of day, and day of week

2. **Spatial patterns:** where violent crime and Medical Incident responses are concentrated in San Francisco

3. **Neighborhood-level response times:** which neighborhoods have faster or slower dispatch-to-on-scene response times

4. **Crime vs response time:** whether neighborhoods with higher violent crime levels also show different EMS response times

### 3.2 Exploring patterns separately

Key plots from this section are later used in the web story

In [ ]:
# ============================================================
# Yearly trends
# We first check how violent crime volume and EMS response time change over time
# ============================================================

# Count violent crime incidents per year
crime_yearly = (
    df_crime_project.groupby('year').size().reset_index(name='count')
)

fig_crime_yr = px.line(
    crime_yearly,
    x='year', y='count',
    markers=True,
    title='Violent Crime Incidents per Year — San Francisco (2016-2025)',
    labels={'year': 'Year', 'count': 'Violent crime incidents'},
    color_discrete_sequence=[COLOR_CRIME],
    template=TEMPLATE,
)

# Mark 2020 because COVID-19 changed city activity and emergency patterns
fig_crime_yr.add_vline(
    x=2020, line_dash='dot', line_color='#888',
    annotation_text='COVID-19', annotation_position='top left'
)
fig_crime_yr.update_traces(line_width=2.5, marker_size=8)
fig_crime_yr.show()
fig_crime_yr.write_html('images/crime_yearly_trend.html')

# Calculate yearly EMS response time statistics
response_yearly = (
    df_ems_project.groupby('year')
    .agg(
        mean_response=('dispatch_to_on_scene_min', 'mean'),
        median_response=('dispatch_to_on_scene_min', 'median'),
        call_count=('Call Number', 'count')
    )
    .reset_index()
)

fig_resp_yr = px.line(
    response_yearly,
    x='year', y='mean_response',
    markers=True,
    title='Mean EMS Response Time by Year — Medical Incidents (2016-2025)',
    labels={'year': 'Year', 'mean_response': 'Mean dispatch-to-on-scene (min)'},
    color_discrete_sequence=[COLOR_EMS],
    template=TEMPLATE,
)

# Mark the same COVID-19 reference point for comparison with the crime trend
fig_resp_yr.add_vline(
    x=2020, line_dash='dot', line_color='#888',
    annotation_text='COVID-19', annotation_position='top left'
)
fig_resp_yr.update_traces(line_width=2.5, marker_size=8)
fig_resp_yr.show()
fig_resp_yr.write_html('images/response_yearly_trend.html')

# Show the yearly response time table used for the plot
display(response_yearly)


,year,mean_response,median_response,call_count
0,2016,7.214,5.400,173794
1,2017,7.038,5.350,181472
2,2018,7.255,5.483,181204
3,2019,7.339,5.550,186563
4,2020,7.486,5.633,163748
5,2021,8.297,6.133,170961
6,2022,8.609,6.350,188665
7,2023,8.443,6.200,200867
8,2024,8.508,6.233,195228
9,2025,8.519,6.317,191299


In [ ]:
# ============================================================
# Hourly patterns
# Here we compare when violent crime happens with how EMS response time changes during the day
# ============================================================

# Count violent crime incidents by hour of day
crime_hour = (
    df_crime_project.groupby('hour').size().reset_index(name='crime_count')
)

# Calculate EMS volume and mean response time by hour of day
ems_hour = (
    df_ems_project.groupby('hour')
    .agg(
        ems_count=('Call Number', 'count'),
        mean_response=('dispatch_to_on_scene_min', 'mean')
    )
    .reset_index()
)

# Join both hourly summaries so they can be plotted together
hourly = crime_hour.merge(ems_hour, on='hour', how='inner')

# Create a dual-axis plot because crime counts and response time have different units
fig_hourly = make_subplots(specs=[[{'secondary_y': True}]])

# Bars show the number of violent crimes by hour
fig_hourly.add_trace(
    go.Bar(
        x=hourly['hour'], y=hourly['crime_count'],
        name='Violent crimes',
        marker_color=COLOR_CRIME, opacity=0.75
    ),
    secondary_y=False
)

# Line shows the mean EMS response time by hour
fig_hourly.add_trace(
    go.Scatter(
        x=hourly['hour'], y=hourly['mean_response'],
        name='Mean EMS response (min)',
        mode='lines+markers',
        marker_color=COLOR_RESPONSE,
        line=dict(width=2.5)
    ),
    secondary_y=True
)

fig_hourly.update_layout(
    title='Violent Crime and EMS Response Time by Hour of Day',
    xaxis_title='Hour of day (0 = midnight)',
    template=TEMPLATE,
    legend=dict(x=0.01, y=0.99)
)
fig_hourly.update_yaxes(title_text='Violent crime incidents (total 2016-2025)', secondary_y=False)
fig_hourly.update_yaxes(title_text='Mean response time (min)', secondary_y=True)

fig_hourly.show()
fig_hourly.write_html('images/hourly_combined.html')

# Print the main hourly values so they can be mentioned in the text if needed
peak_crime_hour = int(hourly.loc[hourly['crime_count'].idxmax(), 'hour'])
peak_response_hour = int(hourly.loc[hourly['mean_response'].idxmax(), 'hour'])
print(f'Peak crime hour:    {peak_crime_hour}:00')
print(f'Slowest EMS hour:   {peak_response_hour}:00')
print(f'Max response time:  {hourly["mean_response"].max():.2f} min')
print(f'Min response time:  {hourly["mean_response"].min():.2f} min')

Peak crime hour:    16:00
Slowest EMS hour:   5:00
Max response time:  8.33 min
Min response time:  7.46 min


In [ ]:
# ============================================================
# Day-of-week patterns
# Here we compare violent crime volume and EMS response time across the week
# ============================================================

# Count violent crime incidents by day of week
crime_dow = (
    df_crime_project.groupby('day_of_week').size()
    .reindex(DOW_ORDER)
    .reset_index(name='crime_count')
)

# Calculate mean EMS response time by day of week
ems_dow = (
    df_ems_project.groupby('day_of_week')
    .agg(mean_response=('dispatch_to_on_scene_min', 'mean'))
    .reindex(DOW_ORDER)
    .reset_index()
)

# Create a dual-axis plot because crime counts and response time have different units
fig_dow = make_subplots(specs=[[{'secondary_y': True}]])

# Bars show the number of violent crimes by weekday
fig_dow.add_trace(
    go.Bar(
        x=crime_dow['day_of_week'], y=crime_dow['crime_count'],
        name='Violent crimes',
        marker_color=COLOR_CRIME, opacity=0.75
    ),
    secondary_y=False
)

# Line shows the mean EMS response time by weekday
fig_dow.add_trace(
    go.Scatter(
        x=ems_dow['day_of_week'], y=ems_dow['mean_response'],
        name='Mean EMS response (min)',
        mode='lines+markers',
        marker_color=COLOR_RESPONSE,
        line=dict(width=2.5)
    ),
    secondary_y=True
)

fig_dow.update_layout(
    title='Violent Crime and EMS Response Time by Day of Week',
    template=TEMPLATE,
    legend=dict(x=0.01, y=0.99)
)
fig_dow.update_yaxes(title_text='Violent crime incidents (total 2016-2025)', secondary_y=False)
fig_dow.update_yaxes(title_text='Mean response time (min)', secondary_y=True)

fig_dow.show()
fig_dow.write_html('images/crime_day_of_week.html')

In [ ]:
# ============================================================
# Crime heatmap by hour and day of week
# This helps us see when violent crime is most concentrated during the week
# ============================================================

# Create a matrix with day of week as rows and hour of day as columns
crime_pivot = (
    df_crime_project
    .groupby(['day_of_week', 'hour'])
    .size()
    .unstack(fill_value=0)
    .reindex(DOW_ORDER)
)

# Plot the matrix as a heatmap
fig_hm = px.imshow(
    crime_pivot,
    title='Violent Crime Intensity — Hour of Day × Day of Week (2016-2025)',
    labels={'x': 'Hour of day', 'y': 'Day of week', 'color': 'Crime count'},
    color_continuous_scale='Reds',
    aspect='auto',
    template=TEMPLATE,
)

# Show every hour on the x-axis
fig_hm.update_layout(xaxis=dict(dtick=1))
fig_hm.show()
fig_hm.write_html('images/crime_hour_heatmap.html')

In [ ]:
# ============================================================
# Geographic heatmaps
# We create interactive maps to see where violent crime and Medical Incident responses are concentrated
# ============================================================

# Center of San Francisco used to initialize the maps
SF_CENTER = [37.7749, -122.4194]

# Create a heatmap with the violent crime locations
crime_pts = df_crime_project[['Latitude', 'Longitude']].dropna().values.tolist()
m_crime = folium.Map(location=SF_CENTER, zoom_start=12)
HeatMap(crime_pts, radius=10, blur=8, min_opacity=0.3).add_to(m_crime)
m_crime.save('images/violent_crime_heatmap.html')
print(f'Crime heatmap saved  —  {len(crime_pts):,} points')

# Create a heatmap with the Medical Incident response locations
ems_pts = df_ems_project[['Latitude', 'Longitude']].dropna().values.tolist()
m_ems = folium.Map(location=SF_CENTER, zoom_start=12)
HeatMap(ems_pts, radius=8, blur=6, min_opacity=0.2).add_to(m_ems)
m_ems.save('images/ems_medical_heatmap.html')
print(f'EMS heatmap saved    —  {len(ems_pts):,} points')

Crime heatmap saved  —  110,062 points
EMS heatmap saved    —  1,833,621 points


In [ ]:
# ============================================================
# Neighborhood-level EMS analysis
# Here we compare how mean response time changes across neighborhoods
# ============================================================

# Column used to identify neighborhoods in the Fire/EMS dataset
NB_COL = 'Neighborhooods - Analysis Boundaries'

# Calculate EMS activity and response time statistics by neighborhood
resp_nb = (
    df_ems_project
    .groupby(NB_COL)
    .agg(
        ems_calls=('Call Number', 'count'),
        mean_response=('dispatch_to_on_scene_min', 'mean'),
        median_response=('dispatch_to_on_scene_min', 'median'),
        als_share=('ALS Unit', 'mean')
    )
    .reset_index()
    .rename(columns={NB_COL: 'neighborhood'})
)

# Keep only neighborhoods with enough observations to make the comparison more stable
resp_nb = resp_nb[resp_nb['ems_calls'] >= 500].copy()

# Sort neighborhoods from slowest to fastest mean response time
resp_nb = resp_nb.sort_values('mean_response', ascending=False)

# Plot mean response time by neighborhood
# Color shows the share of responses handled by ALS units
fig_nb = px.bar(
    resp_nb,
    x='neighborhood', y='mean_response',
    color='als_share',
    color_continuous_scale='Blues',
    hover_data=['ems_calls', 'median_response'],
    title='Mean EMS Response Time by Neighborhood (Medical Incidents, 2016-2025)',
    labels={
        'neighborhood'  : 'Neighborhood',
        'mean_response' : 'Mean response time (min)',
        'als_share'     : 'ALS unit share'
    },
    template=TEMPLATE,
)

fig_nb.update_layout(xaxis_tickangle=-45, height=500)
fig_nb.show()
fig_nb.write_html('images/response_time_by_neighborhood.html')

# Show the neighborhoods with the slowest and fastest response times
print('\nTop 5 slowest-responding neighborhoods:')
display(resp_nb.head(5)[['neighborhood', 'mean_response', 'ems_calls']].reset_index(drop=True))
print('\nTop 5 fastest-responding neighborhoods:')
display(resp_nb.tail(5)[['neighborhood', 'mean_response', 'ems_calls']].reset_index(drop=True))


Top 5 slowest-responding neighborhoods:


,neighborhood,mean_response,ems_calls
0,Lakeshore,10.039,25002
1,Presidio,9.945,7587
2,Visitacion Valley,9.801,24314
3,Treasure Island,9.663,9688
4,Lincoln Park,9.532,915



Top 5 fastest-responding neighborhoods:


,neighborhood,mean_response,ems_calls
0,Castro/Upper Market,7.217,44100
1,Tenderloin,7.184,298598
2,Mission,7.143,177909
3,Western Addition,7.143,66001
4,Nob Hill,6.924,57754


### 3.2.1 What we learned from these first plots

These first plots helped us understand some basic patterns before comparing crime and EMS directly:

- EMS response times became slower over time. From 2016 to 2020, the mean response time was around **7.0-7.5 minutes**. After 2021, it increased and stayed above **8.4 minutes** from 2022 onwards.

- Violent crime and EMS response time do not peak at the same hour. Crime peaks around **16:00**, while the slowest EMS response time appears around **5:00**. Still, the hourly difference in response time is not extremely large, going from around **7.46 to 8.33 minutes**.

- Violent crime is higher on **Friday, Saturday, and Sunday**, especially on Saturday. EMS response time is higher from **Wednesday to Friday**, but lower during the weekend, even though crime volume is high.

- The hour-by-day heatmap shows that crime is especially concentrated during **late-night weekend hours**, mainly around **midnight to 2:00 AM**.

- Response time also changes a lot by neighborhood. The slowest neighborhoods are **Lakeshore**, **Presidio**, **Visitacion Valley**, **Treasure Island**, and **Lincoln Park**, with mean response times close to **9.5-10 minutes**.

- Some central neighborhoods are faster, even with many EMS responses. For example, **Tenderloin** and **Mission** have very high EMS volume but still have average response times around **7.1-7.2 minutes**.

Overall, these plots suggest that response time is not only related to the number of crimes or EMS responses in an area. The location of the neighborhood also seems to matter: some outer neighborhoods are slower, while some central/high-demand areas are faster.

### 3.4 Linking the datasets and modelling response time

#### 3.4.1 The geographic linking challenge

One of the main challenges of the project is that the two datasets **do not use the same geographic units**. The crime dataset includes **Police Districts**, while the Fire/EMS dataset includes **neighborhood names**. These areas do not match perfectly, so we cannot directly compare both datasets using their original geographic labels.

To solve this, we use a **common spatial grid**. We divide the city into grid cells of approximately `1 km × 1 km`, using the latitude and longitude of each record. Each crime incident and each Fire/EMS unit response is assigned to one of these grid cells.

This gives both datasets the same spatial structure, making the comparison more direct


#### 3.4.2 Correlation and regression analysis

After linking both datasets through the spatial grid, we analyse whether areas with more violent crime also tend to have slower or different EMS response times.

First, we calculate the **correlation between violent crime count and mean EMS response time** at the grid-cell level. This gives us a simple first view of whether crime intensity and response time are related spatially.

Then, we fit an **OLS regression model** to study which factors are associated with EMS response time. The goal of this model is not to make perfect predictions, but to better understand the relationship between response time, crime density, EMS demand, and time patterns.

The model is:

```
response_time = β₀ + β₁·hour_sin + β₂·hour_cos
              + β₃·is_weekend + β₄·log(crime_density + 1)
              + β₅·log(ems_demand + 1) + ε
```


The variables are:

- `hour_sin` and `hour_cos`: cyclical encoding of the hour of day. This avoids treating 23:00 and 00:00 as far apart
- `is_weekend`: indicates whether the response happened on Saturday or Sunday
- `log_crime_density`: the log-transformed number of violent crimes in the same grid cell
- `log_ems_demand`: the log-transformed number of Fire/EMS unit responses in the same grid cell. This helps account for areas with generally higher emergency workload

Because the Fire/EMS dataset is very large, we use a **random sample of 150,000 unit responses** for the regression analysis. This makes the model faster to run while still keeping a large sample of the data.

One important thing to keep in mind is that this model **does not prove causality**. If crime density is related to response time, it does not necessarily mean that crime is causing slower responses. Other factors, such as traffic, station location, call priority, or staffing, can also affect how fast units arrive.


In [ ]:
# ============================================================
# Spatial grid and correlation
# Here we put crime and Fire/EMS records into the same grid cells so they can be compared
# ============================================================

# Assign each crime incident and each Fire/EMS response to a grid cell
df_crime_grid = assign_grid(df_crime_project)
df_ems_grid   = assign_grid(df_ems_project)

# Count violent crime incidents in each grid cell
crime_grid = (
    df_crime_grid
    .groupby(['grid_id', 'lat_bin', 'lon_bin'])
    .size()
    .reset_index(name='crime_count')
)

# Calculate Fire/EMS activity and response time statistics in each grid cell
ems_grid = (
    df_ems_grid
    .groupby(['grid_id', 'lat_bin', 'lon_bin'])
    .agg(
        ems_calls=('Call Number', 'count'),
        mean_response=('dispatch_to_on_scene_min', 'mean'),
        median_response=('dispatch_to_on_scene_min', 'median'),
        als_share=('ALS Unit', 'mean')
    )
    .reset_index()
)

# Join both summaries so each row contains crime and Fire/EMS information for the same grid cell
grid_joint = crime_grid.merge(
    ems_grid, on=['grid_id', 'lat_bin', 'lon_bin'], how='inner'
)

# Keep only grid cells with enough crime and Fire/EMS activity to make the averages more stable
grid_joint = grid_joint[
    (grid_joint['crime_count'] >= 10) &
    (grid_joint['ems_calls']   >= 30)
].copy()

print(f'Grid cells (after filtering): {len(grid_joint)}')

# Calculate correlation between violent crime count and mean EMS response time
r_p, p_p = stats.pearsonr(grid_joint['crime_count'], grid_joint['mean_response'])
r_s, p_s = stats.spearmanr(grid_joint['crime_count'], grid_joint['mean_response'])

print(f'\nPearson  r = {r_p:.4f}  (p = {p_p:.3e})')
print(f'Spearman r = {r_s:.4f}  (p = {p_s:.3e})')

# Scatter plot of crime density vs response time by grid cell
# Bubble size shows Fire/EMS volume and color shows ALS unit share
fig_scatter = px.scatter(
    grid_joint,
    x='crime_count', y='mean_response',
    size='ems_calls', color='als_share',
    color_continuous_scale='Blues',
    hover_data=['grid_id', 'median_response', 'crime_count'],
    title='Crime Density vs Mean EMS Response Time — Spatial Grid Cells (~1 km²)',
    labels={
        'crime_count'  : 'Violent crime count (2016-2025)',
        'mean_response': 'Mean EMS response time (min)',
        'als_share'    : 'ALS unit share',
        'ems_calls'    : 'EMS calls'
    },
    template=TEMPLATE,
)

# Add the Pearson correlation directly on the plot
fig_scatter.update_layout(
    annotations=[dict(
        x=0.02, y=0.97, xref='paper', yref='paper',
        text=f'Pearson r = {r_p:.3f}  (p = {p_p:.2e})',
        showarrow=False,
        bgcolor='white', bordercolor='#aaa', borderwidth=1
    )]
)
fig_scatter.show()
fig_scatter.write_html('images/crime_vs_response_scatter.html')

Grid cells (after filtering): 129

Pearson  r = -0.3247  (p = 1.739e-04)
Spearman r = -0.5057  (p = 9.792e-10)


In [ ]:
# ============================================================
# Coarser spatial overview
# Here we use a larger grid to get a simpler view of how crime, EMS demand, and response time relate across the city
# ============================================================

# Use a coarser grid so the bubbles are easier to read
COARSE = 0.02
dc = assign_grid(df_crime_project, gs=COARSE)
de = assign_grid(df_ems_project,   gs=COARSE)

# Count violent crime incidents in each larger grid cell
crime_c = dc.groupby('grid_id').size().reset_index(name='crime_count')

# Calculate EMS volume and mean response time in each larger grid cell
ems_c   = (
    de.groupby('grid_id')
    .agg(
        ems_calls=('Call Number', 'count'),
        mean_response=('dispatch_to_on_scene_min', 'mean'),
    )
    .reset_index()
)

# Join crime and Fire/EMS information for the same larger grid cells
overview_df = crime_c.merge(ems_c, on='grid_id', how='inner')

# Keep only cells with enough activity to make the comparison more stable
overview_df = overview_df[
    (overview_df['crime_count'] >= 50) &
    (overview_df['ems_calls']   >= 200)
].copy()

# Bubble chart of EMS demand and mean response time
# Bubble size and color both show violent crime count
fig_bubble = px.scatter(
    overview_df,
    x='ems_calls', y='mean_response',
    size='crime_count', color='crime_count',
    color_continuous_scale='OrRd',
    hover_data=['grid_id', 'crime_count'],
    title='City Zones: EMS Demand vs Response Time (bubble size = violent crime count)',
    labels={
        'ems_calls'    : 'Total EMS medical calls (2016-2025)',
        'mean_response': 'Mean EMS response time (min)',
        'crime_count'  : 'Violent crimes'
    },
    template=TEMPLATE,
)

fig_bubble.show()
fig_bubble.write_html('images/neighborhood_overview.html')

In [ ]:
# ============================================================
# OLS regression
# Here we model EMS response time using time variables, crime density, and EMS demand
# ============================================================

# Count violent crimes in each grid cell
crime_density = (
    df_crime_grid.groupby('grid_id')
    .size()
    .reset_index(name='crime_count_cell')
)

# Count Fire/EMS responses in each grid cell
ems_demand = (
    df_ems_grid.groupby('grid_id')
    .size()
    .reset_index(name='ems_demand_cell')
)

# Add crime density and EMS demand to each Fire/EMS response record
df_model = (
    df_ems_grid
    .merge(crime_density, on='grid_id', how='left')
    .merge(ems_demand,    on='grid_id', how='left')
)

# Create variables for the regression model
# hour_sin and hour_cos represent the hour of day as a cycle
df_model['hour_sin']          = np.sin(2 * np.pi * df_model['hour'] / 24)
df_model['hour_cos']          = np.cos(2 * np.pi * df_model['hour'] / 24)

# Weekend flag
df_model['is_weekend']        = df_model['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Log variables reduce the effect of very large counts
df_model['log_crime_density'] = np.log1p(df_model['crime_count_cell'].fillna(0))
df_model['log_ems_demand']    = np.log1p(df_model['ems_demand_cell'].fillna(0))

# Use a sample to make the regression faster to run
SAMPLE_N = 150_000
df_sample = (
    df_model
    .dropna(subset=['dispatch_to_on_scene_min', 'crime_count_cell'])
    .sample(SAMPLE_N, random_state=42)
)

# Define model inputs and output
FEATURES = ['hour_sin', 'hour_cos', 'is_weekend', 'log_crime_density', 'log_ems_demand']
X = sm.add_constant(df_sample[FEATURES])
y = df_sample['dispatch_to_on_scene_min']

# Fit the OLS model and print the summary
model = sm.OLS(y, X).fit()
print(model.summary())
print(f'\nModel R² = {model.rsquared:.4f}')
print(f'N        = {len(df_sample):,}')

                               OLS Regression Results                               
Dep. Variable:     dispatch_to_on_scene_min   R-squared:                       0.008
Model:                                  OLS   Adj. R-squared:                  0.008
Method:                       Least Squares   F-statistic:                     232.3
Date:                      Thu, 23 Apr 2026   Prob (F-statistic):          5.49e-248
Time:                              22:54:12   Log-Likelihood:            -4.8463e+05
No. Observations:                    150000   AIC:                         9.693e+05
Df Residuals:                        149994   BIC:                         9.693e+05
Df Model:                                 5                                         
Covariance Type:                  nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------

In [ ]:
# ============================================================
# Regression coefficient chart
# This plot summarizes the estimated effect of each variable in the OLS model
# ============================================================

# Build a table with coefficients and confidence intervals
coef_df = pd.DataFrame({
    'feature'    : model.params.index,
    'coefficient': model.params.values,
    'ci_low'     : model.conf_int()[0].values,
    'ci_high'    : model.conf_int()[1].values,
}).query("feature != 'const'")

# Replace variable names with simpler labels for the plot
FEATURE_LABELS = {
    'hour_sin'          : 'Hour (sine)',
    'hour_cos'          : 'Hour (cosine)',
    'is_weekend'        : 'Weekend',
    'log_crime_density' : 'Log crime density (grid)',
    'log_ems_demand'    : 'Log EMS demand (grid)',
}
coef_df['label'] = coef_df['feature'].map(FEATURE_LABELS)

# Create the figure
fig_coef = go.Figure()

# Add one point per variable, together with its 95% confidence interval
for _, row in coef_df.iterrows():
    color = COLOR_CRIME if row['coefficient'] > 0 else COLOR_EMS
    fig_coef.add_trace(go.Scatter(
        x=[row['coefficient']],
        y=[row['label']],
        mode='markers',
        marker=dict(color=color, size=12),
        error_x=dict(
            type='data', symmetric=False,
            array=[row['ci_high'] - row['coefficient']],
            arrayminus=[row['coefficient'] - row['ci_low']],
            visible=True
        ),
        name=row['label'],
        showlegend=False
    ))

# Add a vertical reference line at zero
fig_coef.add_vline(x=0, line_dash='dash', line_color='#999')

# Final layout of the coefficient plot
fig_coef.update_layout(
    title='OLS Regression: Predictors of EMS Response Time (coeff + 95% CI)',
    xaxis_title='Coefficient (minutes)',
    yaxis_title='',
    template=TEMPLATE,
    height=400,
)

fig_coef.show()
fig_coef.write_html('images/regression_coefficients.html')

---
## 4. Genre

### 4.1 Which genre did we use?

Following Segel & Heer (2010), our initial idea was to use a **Martini Glass** structure. This made sense at the beginning because we wanted to guide the reader through the main findings first, and then allow more open exploration through interactive visualizations.

However, as the website developed, the final result became closer to a **Magazine Style** data story. The website is structured as a scrolling article where text, key numbers, maps, and interactive visualizations are combined to guide the reader through the analysis.

This genre fits our project because the datasets are large and not directly linked, so the reader needs context before interpreting the visualizations. A magazine-style format lets us introduce the research question, explain the datasets, show the main spatial and temporal patterns, and then finish with the key findings. 

For this reason, a **guided structure** fits our data story better, so the reader can explore details without losing the main narrative. The website still includes interactive elements.

### 4.2 Visual Narrative tools (Segel & Heer, Figure 7)

| Category | Tool used | How |
|---|---|---|
| **Visual Structuring** | Consistent Visual Platform | Same colors and layout style across the website |
| **Highlighting** | Feature Distinction | Red = crime, blue = EMS, amber = response time |
| **Highlighting** | Zooming | Interactive maps and plots allow closer inspection |
| **Transition Guidance** | Continuity Editing | Each section leads naturally into the next one |

### 4.3 Narrative Structure tools (Segel & Heer, Figure 7)

| Category | Tool used | How |
|---|---|---|
| **Ordering** | Linear | The reader follows a fixed scroll-based story |
| **Interactivity** | Hover Highlighting / Details | Hover shows values, locations, and extra context |
| **Interactivity** | Very Limited Interactivity | Interaction supports exploration but does not replace the guided story |
| **Messaging** | Captions / Headlines | Titles and captions explain each section |
| **Messaging** | Accompanying Article | The text explains the context around the charts |
| **Messaging** | Summary / Synthesis | The key findings section summarizes the final message |


---

## 5. Visualizations

These are the visualizations that we chose for the final data story website. They are ordered as they appear on the web:

### 5.1 Geographic heatmaps (Folium / Leaflet)

We use heatmaps to show where **violent crime** and **Medical Incident responses** are concentrated in San Francisco.

This works well because both datasets include point locations. Heatmaps make hotspots easy to see without depending on neighborhood or police district boundaries, which are different in the two datasets.

We use separate interactive maps for crime and Fire/EMS activity so the reader can compare the hotspots visually.

### 5.2 Bar chart: EMS response time by neighborhood

We use a sorted bar chart to compare **average EMS response time** across neighborhoods.

This makes it easy to see which neighborhoods have slower or faster dispatch-to-on-scene times. Sorting the bars helps the reader understand the ranking quickly.

We also color the bars by **ALS unit share**, which shows the proportion of responses handled by Advanced Life Support units. This adds context about the type of medical response in each neighborhood.

### 5.3 Scatter plot: crime density vs response time

This visualization directly connects to our main research question.

We use a scatter plot to compare **violent crime density** and **mean EMS response time** at the grid-cell level. Each point represents one spatial grid cell, so we can see whether areas with more violent crime also tend to have longer response times.

The Pearson r annotation gives the reader a simple quantitative summary of the relationship.

### 5.4 Dual-axis line/bar chart: hour of day

We use a dual-axis chart to compare **violent crime volume** and **EMS response time** across the 24 hours of the day.

This works well because the two variables have different units and scales. Bars show the number of violent crimes, while the line shows the mean EMS response time.

Using the same time axis makes it easier to compare the daily rhythm of crime with the daily rhythm of EMS response time.

### 5.5 Heatmap: hour × day of week

We use a heatmap to show how violent crime changes by hour of day and day of week.

This is useful because a simple hourly chart would only show the average daily pattern. The heatmap also shows whether the pattern changes between weekdays and weekends.

It helps highlight when violent crime is most concentrated, especially during late-night weekend hours.

### 5.6 Line charts: yearly trends

We use line charts to show how violent crime and EMS response time changed from **2016 to 2025**.

This works well because the data is ordered over time. The charts make it easier to see long-term changes, sudden drops, and periods of increase.

We also mark 2020 because **COVID-19** changed both city activity and emergency response conditions.

### 5.7 Regression coefficient chart

We use a coefficient chart to summarize the **OLS regression results**.

This chart shows the estimated effect of each variable on EMS response time, together with its confidence interval. We found it easier to read than a full regression table.

It helps us see which variables are associated with longer or shorter response times, and whether the uncertainty range crosses zero.

---

## 6. Discussion

### 6.1 What went well?

- The **spatial grid** worked well for linking the two datasets. The crime data and the Fire/EMS data use different geographic labels, so the grid gave us a common way to compare them. This was especially useful for the scatter plot and the regression.

- The **temporal analysis** was useful because it showed patterns that were easy to understand. The hourly and day-of-week plots helped us see when crime is more concentrated and how EMS response time changes during the day and week.

- The **regression** helped us go beyond only looking at the plots. It allowed us to check whether crime density was still related to response time when also including EMS demand, hour of day, and weekend effects. We used **OLS** because it is easier to interpret.

- The **visual design** worked quite well. We kept the same colors throughout the notebook and website: red for crime, blue for Fire/EMS, and amber for response time. This made the story easier to follow.

- The **magazine-style website structure** also worked well for our audience. It let us explain the project step by step, instead of showing all the charts without context.

### 6.2 What is still missing? What could be improved?

- The main limitation is that we do not match **individual crime incidents** with **individual EMS responses**. Our analysis is based on grouping the data by space and time. A stronger version would try to match crimes with nearby EMS responses, for example within 500 meters and 30 minutes. This would give a more direct answer, but it would also be much harder to validate

- **Crime density** and slower EMS response may both be related to other factors. For example, income, housing density, traffic, station location, infrastructure, or staffing could also affect response time. Our regression controls for EMS demand, hour of day, and weekends, but it does not include socioeconomic or transport variables

- The Fire/EMS category could be more detailed. **Medical Incident** is a broad category, and not all medical calls are related to violence or serious injury. A future version could separate calls by severity, final disposition, transport type, or priority level

- The **geographic visualizations** could also be improved. The heatmaps are useful for showing hotspots, but a choropleth map with official neighborhood boundaries could be easier to understand for a general audience. For example, it could include layers for crime density and response time

- We mark **COVID-19 (2020)** in the yearly plots, but we do not analyse this period in detail. It could be interesting to study 2020 separately, because city activity, crime patterns, and emergency response conditions all changed during that period

- The **model** could be extended. We used OLS regression because it is easy to interpret, but more complex models such as random forests might predict response time better. The downside is that they would also be harder to explain clearly in the website

---

## 7. Contributions

We both worked on the project and reviewed the final notebook and website. Ignacio Ripoll created the first working version of the analysis and visualizations, including data filtering, response time calculation, the spatial grid, and regression analysis. Raquel Ruiz focused on refining and improving that version, writing the explanations, checking consistency between the notebook and the website, and adapting the visualizations and narrative for the final data story. The final version is worked by both of us, we both discussed the results and understand the full project.


---

## References


- DataSF. *Police Department Incident Reports: Historical 2003 to May 2018*. City and County of San Francisco Open Data Portal.  
  https://data.sfgov.org/Public-Safety/Police-Department-Incident-Reports-Historical-2003/tmnf-yvry/about_data

- DataSF. *Police Department Incident Reports: 2018 to Present*. City and County of San Francisco Open Data Portal.  
  https://data.sfgov.org/Public-Safety/Police-Department-Incident-Reports-2018-to-Present/wg3w-h783/about_data

- DataSF. *Fire Department and Emergency Medical Services Dispatched Calls for Service*. City and County of San Francisco Open Data Portal.  
  https://data.sfgov.org/Public-Safety/Fire-Department-and-Emergency-Medical-Services-Dis/nuek-vuh3/about_data

- Segel, E., & Heer, J. (2010). *Narrative Visualization: Telling Stories with Data*. IEEE Transactions on Visualization and Computer Graphics, 16(6), 1139–1148.


**Web story:** [index.html](index.html)